# Proyecto 3: Reconocimiento de ASL, versión con CUDA

Juan Diego Solís Martínez - 23720  
Víctor Manuel Pérez Chávez - 23731

Versión paralela de `proyecto3_asl_secuencial.ipynb`. Usa los mismos datos, la misma partición, los mismos modelos y los mismos hiperparámetros, y separa el trabajo en dos fases según el hardware que conviene a cada una.

| Fase | Sección | Dispositivo |
|---|---|---|
| Fase 1: preparación de datos | 1 y 2 | CPU, todos los núcleos |
| Fase 2: matemática de la red neuronal | 3 a 5 | GPU con CUDA |
| Modelo clásico | 6 | CPU, todos los núcleos |
| Exactitud, comparación y speedup | 7 y 8 | |

Los tiempos secuenciales no se vuelven a medir aquí: se leen de `results/`, donde los dejó el notebook secuencial ejecutado en esta misma máquina.

## 0. Configuración

Mismos parámetros que la versión secuencial. La única diferencia es `N_JOBS`, que pasa de 1 a todos los núcleos lógicos, y que la GPU queda visible.

In [1]:
import os
import time

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"  # TensorFlow no reserva toda la memoria: deja espacio a Numba

MODO = "benchmark"                      # "benchmark" o "completo"
N_REPETICIONES = 3                      # solo aplica en benchmark
IMGS_POR_CLASE_BENCHMARK = 800
EPOCAS_BENCHMARK = 5
N_JOBS = os.cpu_count()                 # procesos para la preparación de datos y el modelo clásico
SEED = 42

# Las variables de entorno ASL_* cambian la configuración sin editar el notebook
MODO = os.environ.get("ASL_MODO", MODO)
N_REPETICIONES = int(os.environ.get("ASL_REPETICIONES", N_REPETICIONES))
IMGS_POR_CLASE_BENCHMARK = int(os.environ.get("ASL_IMGS_POR_CLASE", IMGS_POR_CLASE_BENCHMARK))
EPOCAS_BENCHMARK = int(os.environ.get("ASL_EPOCAS", EPOCAS_BENCHMARK))
N_JOBS = int(os.environ.get("ASL_N_JOBS", N_JOBS))
DIR_SALIDA = os.environ.get("ASL_SALIDA", ".")  # donde se crean results/, figures/ y evidencias/

T_INICIO_NOTEBOOK = time.perf_counter()

In [2]:
import sys
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/JuanDsm04/pdc-final-project.git"

if EN_COLAB and not Path("artifacts").exists():
    !git clone -q {REPO_URL}
    %cd pdc-final-project
    !pip install -q kagglehub "numba-cuda[cu12]"

In [3]:
import re
import json
import math
import random
import platform
import warnings
import subprocess
from contextlib import contextmanager

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import psutil
from PIL import Image
from joblib import Parallel, delayed

from skimage.feature import hog
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score

def precargar_cusolver():
    """En Linux, TensorFlow 2.21 no encuentra el cuSOLVER que instala pip y desactiva la GPU sin error.
    Cargarlo antes de importar TensorFlow lo deja disponible."""
    import ctypes, glob, sysconfig
    carpeta = os.path.join(sysconfig.get_paths()["purelib"], "nvidia", "cusolver", "lib")
    for lib in sorted(glob.glob(os.path.join(carpeta, "libcusolver.so.*"))):
        ctypes.CDLL(lib, mode=ctypes.RTLD_GLOBAL)

if platform.system() == "Linux":
    precargar_cusolver()

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, Callback

import numba
from numba import cuda, float32

from monitor_recursos import MonitorRecursos

warnings.filterwarnings("ignore")
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

TARGET_SIZE = (64, 64)
N_CLASSES = 29
BATCH_SIZE = 64
REPS = N_REPETICIONES if MODO == "benchmark" else 1
CPU_N = f"CPU x{N_JOBS}"

ROOT = Path.cwd()
SALIDA = Path(DIR_SALIDA).resolve()
RESULTS_DIR = SALIDA / "results"
FIGURES_DIR = SALIDA / "figures"
EVIDENCIAS_DIR = SALIDA / "evidencias"
for carpeta in (RESULTS_DIR, FIGURES_DIR, EVIDENCIAS_DIR):
    carpeta.mkdir(parents=True, exist_ok=True)

AZUL, NARANJA, VERDE = "#2a78d6", "#eb6834", "#1baf7a"
TINTA, TINTA2, GRIS, REJILLA, FONDO = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#fcfcfb"
plt.rcParams.update({
    "figure.facecolor": FONDO, "axes.facecolor": FONDO, "savefig.facecolor": FONDO,
    "axes.edgecolor": "#c3c2b7", "axes.labelcolor": TINTA2, "text.color": TINTA,
    "xtick.color": TINTA2, "ytick.color": TINTA2, "grid.color": REJILLA, "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False, "axes.axisbelow": True,
    "axes.titlelocation": "left", "axes.titlesize": 12, "font.size": 9,
    "axes.prop_cycle": plt.cycler(color=[AZUL, NARANJA, VERDE]),
})

### 0.1 Configuración de CUDA

TensorFlow ejecuta las capas de la red con cuDNN y CUDA. Numba compila a CUDA los kernels propios de la sección 5. Los dos usan la misma GPU, así que TensorFlow se configura para pedir memoria solo cuando la necesita.

In [4]:
GPUS = tf.config.list_physical_devices("GPU")
assert GPUS, "TensorFlow no detecta ninguna GPU: revise el driver y las librerías de CUDA"
for gpu in GPUS:
    tf.config.experimental.set_memory_growth(gpu, True)

DETALLE_GPU = tf.config.experimental.get_device_details(GPUS[0])
BUILD_TF = tf.sysconfig.get_build_info()
print("TensorFlow:", GPUS[0].name, "|", DETALLE_GPU.get("device_name"),
      "| compute capability", DETALLE_GPU.get("compute_capability"))
print(f"TensorFlow compilado con CUDA {BUILD_TF.get('cuda_version')} y cuDNN {BUILD_TF.get('cudnn_version')}\n")
assert cuda.detect(), "Numba no encuentra una GPU compatible con CUDA"

TensorFlow: /physical_device:GPU:0 | NVIDIA GeForce GTX 1650 | compute capability (7, 5)
TensorFlow compilado con CUDA 12.5.1 y cuDNN 9

Found 1 CUDA devices
id 0    NVIDIA GeForce GTX 1650                              [SUPPORTED]
                      Compute Capability: 7.5
                           PCI Device ID: 0
                              PCI Bus ID: 1
                                    UUID: GPU-fb4e5528-5a46-9374-3681-aeff2cef51e2
                                Watchdog: Disabled
             FP32/FP64 Performance Ratio: 32
                            FP16 Support: Yes
                            BF16 Support: No
                             FP8 Support: No
Summary:
	1/1 devices are supported


### 0.2 Hardware y entorno

Se guarda en `results/hardware_cuda_<modo>.json`.

In [5]:
GPU_NUMBA = cuda.get_current_device().name

def info_hardware():
    info = {
        "fecha": time.strftime("%Y-%m-%d %H:%M:%S"),
        "entorno": "Google Colab" if EN_COLAB else "Local",
        "sistema": f"{platform.system()} {platform.release()}",
        "cpu": platform.processor() or platform.machine(),
        "nucleos_fisicos": psutil.cpu_count(logical=False),
        "nucleos_logicos": psutil.cpu_count(logical=True),
        "ram_gb": round(psutil.virtual_memory().total / 1e9, 1),
        "python": platform.python_version(),
        "tensorflow": tf.__version__,
        "numpy": np.__version__,
        "numba": numba.__version__,
        "cuda_tensorflow": BUILD_TF.get("cuda_version"),
        "cudnn_tensorflow": BUILD_TF.get("cudnn_version"),
        "dispositivos_tf": [d.name for d in tf.config.list_physical_devices()],
        "gpu_numba": GPU_NUMBA.decode() if isinstance(GPU_NUMBA, bytes) else str(GPU_NUMBA),
        "modo": MODO,
        "repeticiones": REPS,
        "imgs_por_clase_benchmark": IMGS_POR_CLASE_BENCHMARK,
        "epocas_benchmark": EPOCAS_BENCHMARK,
        "n_jobs": N_JOBS,
    }
    cpuinfo = Path("/proc/cpuinfo")
    if cpuinfo.exists():
        modelos = re.findall(r"model name\s*:\s*(.+)", cpuinfo.read_text())
        if modelos:
            info["cpu"] = modelos[0]
    try:
        salida = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
            capture_output=True, text=True, timeout=10,
        ).stdout.strip()
        info["gpu"] = salida or "No detectada"
    except (FileNotFoundError, subprocess.TimeoutExpired):
        info["gpu"] = "No detectada"
    return info

HARDWARE = info_hardware()
(RESULTS_DIR / f"hardware_cuda_{MODO}.json").write_text(json.dumps(HARDWARE, indent=2, ensure_ascii=False))
pd.Series(HARDWARE, dtype=object).to_frame("valor")

,valor
fecha,2026-11-18 04:47:55
entorno,Local
sistema,Linux 7.2.15-arch1-1
cpu,AMD Ryzen 7 5800HS with Radeon Graphics
nucleos_fisicos,8
nucleos_logicos,16
ram_gb,24.6
python,3.12.15
tensorflow,2.21.0
numpy,2.4.6


### 0.3 Cronometraje

Mismo formato que la versión secuencial, en `results/tiempos_cuda_<modo>.csv`. La columna `dispositivo` indica dónde corrió cada fase. Los tiempos son de reloj, así que los de la GPU ya incluyen las transferencias entre CPU y GPU; la sección 3 las mide además por separado. `MonitorRecursos` registra el uso de cada núcleo y de la GPU.

In [6]:
REGISTRO_TIEMPOS = []
CSV_TIEMPOS = RESULTS_DIR / f"tiempos_cuda_{MODO}.csv"
MONITOR = MonitorRecursos("cuda", MODO, RESULTS_DIR, EVIDENCIAS_DIR).iniciar()

def registrar(fase, modelo, repeticion, segundos, n_imagenes=None, epoca=None, dispositivo="GPU"):
    REGISTRO_TIEMPOS.append({
        "parte": "cuda", "modo": MODO, "fase": fase, "modelo": modelo,
        "repeticion": repeticion, "epoca": epoca, "segundos": segundos,
        "n_imagenes": n_imagenes, "dispositivo": dispositivo,
    })
    pd.DataFrame(REGISTRO_TIEMPOS).to_csv(CSV_TIEMPOS, index=False)

@contextmanager
def cronometro(fase, modelo="-", repeticion=1, n_imagenes=None, dispositivo="GPU"):
    with MONITOR.fase(fase if modelo == "-" else f"{fase} | {modelo}"):
        t0 = time.perf_counter()
        yield
        segundos = time.perf_counter() - t0
    registrar(fase, modelo, repeticion, segundos, n_imagenes, dispositivo=dispositivo)
    print(f"[{fase} | {modelo} | rep {repeticion} | {dispositivo}] {segundos:.2f} s")

class TiempoPorEpoca(Callback):
    def __init__(self, modelo, repeticion=1, dispositivo="GPU"):
        super().__init__()
        self.modelo, self.repeticion, self.dispositivo = modelo, repeticion, dispositivo

    def on_epoch_begin(self, epoch, logs=None):
        self.t0 = time.perf_counter()

    def on_epoch_end(self, epoch, logs=None):
        registrar("epoca", self.modelo, self.repeticion, time.perf_counter() - self.t0,
                  epoca=epoch + 1, dispositivo=self.dispositivo)

## 1. Datos y partición

Mismo dataset, misma partición de `artifacts/splits.csv` y misma submuestra de entrenamiento que la versión secuencial. El análisis exploratorio no se repite.

In [7]:
def localizar_dataset():
    local = ROOT / "archive"
    if local.exists():
        return local
    import kagglehub
    return Path(kagglehub.dataset_download("grassknoted/asl-alphabet"))

DATA_DIR = localizar_dataset()
TRAIN_DIR = DATA_DIR / "asl_alphabet_train"
if (TRAIN_DIR / "asl_alphabet_train").is_dir():
    TRAIN_DIR = TRAIN_DIR / "asl_alphabet_train"

with cronometro("inventario", dispositivo="CPU"):
    CLASSES = sorted(d.name for d in TRAIN_DIR.iterdir() if d.is_dir())
    files_per_class = {cls: sorted((TRAIN_DIR / cls).iterdir()) for cls in CLASSES}

print(f"{len(CLASSES)} clases, {sum(len(f) for f in files_per_class.values()):,} imágenes en {TRAIN_DIR}")

[inventario | - | rep 1 | CPU] 0.78 s
29 clases, 87,000 imágenes en /home/vic/Code/pdc-final-project/archive/asl_alphabet_train/asl_alphabet_train


In [8]:
SPLITS_CSV = ROOT / "artifacts" / "splits.csv"

def generar_particion():
    df_all = pd.DataFrame(
        [{"rel": f"{cls}/{f.name}", "clase": cls} for cls, files in files_per_class.items() for f in files]
    )
    train, temp = train_test_split(df_all, test_size=0.30, stratify=df_all["clase"], random_state=SEED)
    val, test = train_test_split(temp, test_size=0.50, stratify=temp["clase"], random_state=SEED)
    return pd.concat([train.assign(split="train"), val.assign(split="val"), test.assign(split="test")])

with cronometro("particion", dispositivo="CPU"):
    splits_df = pd.read_csv(SPLITS_CSV) if SPLITS_CSV.exists() else generar_particion()
    splits_df["path"] = [str(TRAIN_DIR / rel) for rel in splits_df["rel"]]
    train_df, val_df, test_df = (
        splits_df.loc[splits_df["split"] == s, ["path", "clase"]].reset_index(drop=True)
        for s in ("train", "val", "test")
    )

label_map = {cls: i for i, cls in enumerate(CLASSES)}
idx_to_class = {i: cls for cls, i in label_map.items()}

if MODO == "benchmark":
    train_fit_df = (
        train_df.groupby("clase", group_keys=False)
        .sample(n=IMGS_POR_CLASE_BENCHMARK, random_state=SEED)
        .reset_index(drop=True)
    )
else:
    train_fit_df = train_df

print(f"Modo: {MODO} | train: {len(train_fit_df):,} | val: {len(val_df):,} | test: {len(test_df):,} | repeticiones: {REPS}")

[particion | - | rep 1 | CPU] 0.53 s
Modo: completo | train: 60,900 | val: 13,050 | test: 13,050 | repeticiones: 1


# Fase 1: preparación de datos en CPU multinúcleo

## 2. Preprocesamiento paralelo

Abrir un archivo, decodificar el JPEG y redimensionarlo es trabajo con ramas y acceso a disco, distinto para cada imagen. Eso lo hacen bien los núcleos del CPU y mal una GPU. Como cada imagen es independiente de las demás, la lista de rutas se reparte en bloques entre `N_JOBS` procesos.

Dos cambios respecto a la versión secuencial:

- **Una sola vez.** Las imágenes se preparan una vez y quedan en memoria. La versión secuencial las vuelve a leer del disco en cada época.
- **`uint8` en lugar de `float32`.** La división entre 255 se deja para la GPU. Así cada imagen ocupa 4 veces menos en memoria y en la transferencia hacia la GPU.

In [9]:
def preprocess_image(path, size=TARGET_SIZE):
    with Image.open(path) as img:
        img = img.convert("RGB").resize(size, Image.BICUBIC)
    return np.array(img, dtype=np.float32) / 255.0

def preprocess_uint8(path, size=TARGET_SIZE):
    with Image.open(path) as img:
        return np.asarray(img.convert("RGB").resize(size, Image.BICUBIC), dtype=np.uint8)

def preprocesar_bloque(paths):
    return np.stack([preprocess_uint8(p) for p in paths])

def cargar_paralelo(df, n_jobs=N_JOBS, tam_bloque=256):
    paths = df["path"].tolist()
    bloques = [paths[i:i + tam_bloque] for i in range(0, len(paths), tam_bloque)]
    partes = Parallel(n_jobs=n_jobs)(delayed(preprocesar_bloque)(b) for b in bloques)
    return np.concatenate(partes), df["clase"].map(label_map).to_numpy(dtype=np.int32)

def calentar_pool(n_jobs):
    paths = train_fit_df["path"].tolist()
    Parallel(n_jobs=n_jobs)(delayed(preprocesar_bloque)(paths[i:i + 4]) for i in range(0, 4 * n_jobs, 4))

# La versión uint8 seguida de /255 debe dar exactamente lo mismo que el preprocesamiento secuencial
muestra = train_fit_df["path"].sample(min(200, len(train_fit_df)), random_state=SEED)
identicas = all(np.array_equal(preprocess_uint8(p).astype(np.float32) / 255.0, preprocess_image(p)) for p in muestra)
print(f"Comprobación en {len(muestra)} imágenes: resultado idéntico al secuencial = {identicas}")
assert identicas

Comprobación en 200 imágenes: resultado idéntico al secuencial = True


### 2.1 Carga medida

El arranque de los procesos se mide aparte, porque ocurre una sola vez. Después se repite la carga del conjunto de entrenamiento `REPS` veces, igual que en la versión secuencial, y se cargan validación y test.

In [10]:
with cronometro("arranque_pool", "paralelo", 1, dispositivo=CPU_N):
    calentar_pool(N_JOBS)

for rep in range(1, REPS + 1):
    X_train = y_train = None
    with cronometro("preprocesamiento", "paralelo", rep, len(train_fit_df), CPU_N):
        X_train, y_train = cargar_paralelo(train_fit_df)

with cronometro("preprocesamiento", "val", 1, len(val_df), CPU_N):
    X_val, y_val = cargar_paralelo(val_df)
with cronometro("preprocesamiento", "test", 1, len(test_df), CPU_N):
    X_test, y_test = cargar_paralelo(test_df)

Y_train = keras.utils.to_categorical(y_train, N_CLASSES)
Y_val = keras.utils.to_categorical(y_val, N_CLASSES)

for nombre, X in [("train", X_train), ("val", X_val), ("test", X_test)]:
    print(f"X_{nombre}: {X.shape} {X.dtype} ({X.nbytes / 1e6:.0f} MB)")

[arranque_pool | paralelo | rep 1 | CPU x16] 0.32 s


[preprocesamiento | paralelo | rep 1 | CPU x16] 5.09 s


[preprocesamiento | val | rep 1 | CPU x16] 1.11 s


[preprocesamiento | test | rep 1 | CPU x16] 1.12 s
X_train: (60900, 64, 64, 3) uint8 (748 MB)
X_val: (13050, 64, 64, 3) uint8 (160 MB)
X_test: (13050, 64, 64, 3) uint8 (160 MB)


### 2.2 Escalabilidad con el número de procesos

Solo en `benchmark`: el mismo conjunto de entrenamiento con 1, 2, 4, 8 y más procesos. Con 1 proceso `joblib` ejecuta el bucle en el proceso principal, que equivale a la carga secuencial.

In [11]:
if MODO == "benchmark":
    procesos = sorted({n for n in (1, 2, 4, 8, 16, 32) if n < N_JOBS} | {N_JOBS})
    for n in procesos:
        calentar_pool(n)
        with cronometro("escalabilidad_preprocesamiento", f"{n} procesos", 1, len(train_fit_df), f"CPU x{n}"):
            cargar_paralelo(train_fit_df, n_jobs=n)

    esc = pd.DataFrame(REGISTRO_TIEMPOS).query("fase == 'escalabilidad_preprocesamiento'").copy()
    esc["procesos"] = procesos
    esc["speedup"] = esc["segundos"].iloc[0] / esc["segundos"]
    esc["eficiencia"] = esc["speedup"] / esc["procesos"]
    esc[["procesos", "segundos", "speedup", "eficiencia"]].to_csv(
        RESULTS_DIR / "escalabilidad_preprocesamiento.csv", index=False)
    print(esc[["procesos", "segundos", "speedup", "eficiencia"]].round(3).to_string(index=False))

    fisicos = psutil.cpu_count(logical=False)
    fig, ax = plt.subplots(figsize=(7, 4.2))
    ax.plot(procesos, procesos, color=GRIS, linewidth=1)
    ax.text(procesos[-1], procesos[-1], "ideal ", ha="right", va="center", color=GRIS)
    ax.plot(esc["procesos"], esc["speedup"], color=AZUL, linewidth=2, marker="o", markersize=7)
    for n, s in zip(esc["procesos"], esc["speedup"]):
        ax.annotate(f"{s:.1f}x", (n, s), textcoords="offset points", xytext=(0, 8), ha="center", color=TINTA2)
    if fisicos in procesos:
        ax.axvline(fisicos, color=REJILLA, linewidth=1)
        ax.text(fisicos, 0.3, f" {fisicos} núcleos físicos", color=TINTA2, va="bottom")
    ax.set_xticks(procesos)
    ax.set_xlabel("Procesos")
    ax.set_ylabel("Speedup respecto a 1 proceso")
    ax.set_title(f"Preprocesamiento de {len(train_fit_df):,} imágenes: speedup según el número de procesos")
    ax.grid(True, axis="y")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "escalabilidad_preprocesamiento.png", dpi=150, bbox_inches="tight")
    plt.show()
    calentar_pool(N_JOBS)

# Fase 2: matemática de la red neuronal en GPU

## 3. Transferencias entre CPU y GPU

Los datos preparados viven en la memoria del CPU. En cada paso de entrenamiento un lote de 64 imágenes `uint8` viaja a la GPU (H2D) y al predecir vuelven las probabilidades (D2H). Ese costo ya está dentro de los tiempos de entrenamiento e inferencia; aquí se mide aislado para saber cuánto pesa.

In [12]:
def medir_h2d(X, lote=BATCH_SIZE):
    with tf.device("/GPU:0"):
        t0 = time.perf_counter()
        for i in range(0, len(X), lote):
            en_gpu = tf.identity(X[i:i + lote])
        tf.reduce_sum(tf.cast(en_gpu, tf.float32)).numpy()   # espera a que termine la última copia
        return time.perf_counter() - t0, en_gpu.device

def medir_d2h(n, lote=BATCH_SIZE):
    with tf.device("/GPU:0"):
        probs = tf.random.uniform((n, N_CLASSES))
    t0 = time.perf_counter()
    for i in range(0, n, lote):
        probs[i:i + lote].numpy()
    return time.perf_counter() - t0

medir_h2d(X_train[:BATCH_SIZE * 4])
for rep in range(1, REPS + 1):
    segundos, dispositivo = medir_h2d(X_train)
    registrar("transferencia_h2d", "train_uint8", rep, segundos, len(X_train))
    registrar("transferencia_d2h", "predicciones_test", rep, medir_d2h(len(X_test)), len(X_test))

transf = pd.DataFrame(REGISTRO_TIEMPOS)
transf = transf[transf["fase"].str.startswith("transferencia")].groupby("fase")["segundos"].mean()
mb = X_train.nbytes / 1e6
print(f"Destino de los lotes: {dispositivo}")
print(f"H2D de train por lotes de {BATCH_SIZE}: {transf['transferencia_h2d']:.3f} s por época "
      f"({mb:.0f} MB, {mb / transf['transferencia_h2d']:.0f} MB/s)")
print(f"D2H de las predicciones de test: {transf['transferencia_d2h']:.3f} s")

Destino de los lotes: /job:localhost/replica:0/task:0/device:GPU:0
H2D de train por lotes de 64: 0.277 s por época (748 MB, 2706 MB/s)
D2H de las predicciones de test: 0.055 s


## 4. Entrenamiento en GPU con TensorFlow

Los constructores de los modelos, el optimizador, la pérdida, el tamaño de lote y los callbacks son los de la versión secuencial. Cambian dos cosas: los datos llegan desde memoria en lugar de leerse del disco, y la normalización a [0, 1] se hace en la GPU con una capa `Rescaling` antepuesta al modelo.

### 4.1 Funciones de entrenamiento y evaluación

In [13]:
EPOCAS_LAB = {"cnn_shallow": 30, "cnn_medium": 40, "fc_base": 40}

def epocas(nombre):
    return EPOCAS_LAB[nombre.replace("_cpu_memoria", "")] if MODO == "completo" else EPOCAS_BENCHMARK

def get_callbacks(nombre, repeticion=1, dispositivo="GPU"):
    callbacks = [
        TiempoPorEpoca(nombre, repeticion, dispositivo),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4, min_lr=1e-6, verbose=1),
    ]
    if MODO == "completo":
        callbacks.append(EarlyStopping(monitor="val_accuracy", patience=8, restore_best_weights=True, verbose=1))
    return callbacks

def con_normalizacion(modelo):
    """Antepone la división entre 255 para que también se calcule en el dispositivo del modelo."""
    entrada = keras.Input(shape=(*TARGET_SIZE, 3), dtype="uint8")
    x = layers.Rescaling(1.0 / 255.0)(entrada)
    return keras.Model(entrada, modelo(x), name=modelo.name + "_uint8")

def compilar(modelo, learning_rate):
    completo = con_normalizacion(modelo)
    completo.compile(optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
                     loss="categorical_crossentropy", metrics=["accuracy"])
    return completo

def entrenar(model, nombre, repeticion=1, verbose=1, dispositivo="GPU"):
    with cronometro("entrenamiento", nombre, repeticion, len(X_train), dispositivo):
        return model.fit(X_train, Y_train, batch_size=BATCH_SIZE, shuffle=True, epochs=epocas(nombre),
                         validation_data=(X_val, Y_val),
                         callbacks=get_callbacks(nombre, repeticion, dispositivo), verbose=verbose)

def predecir(model, X, fase=None, nombre="-", repeticion=1):
    if fase is None:
        return model.predict(X, batch_size=BATCH_SIZE, verbose=0).argmax(axis=1)
    with cronometro(fase, nombre, repeticion, len(X)):
        return model.predict(X, batch_size=BATCH_SIZE, verbose=0).argmax(axis=1)

def plot_history(history, title):
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    for ax, metrica, etiqueta in [(axes[0], "accuracy", "Accuracy"), (axes[1], "loss", "Loss")]:
        ax.plot(history.history[metrica], label="Train", linewidth=2)
        ax.plot(history.history[f"val_{metrica}"], label="Val", linewidth=2)
        ax.set_title(f"{title}: {etiqueta} por época")
        ax.set_xlabel("Época")
        ax.set_ylabel(etiqueta)
        ax.legend()
        ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

### 4.2 Modelo 1: CNN Shallow

In [14]:
def build_cnn_shallow(input_shape=(64, 64, 3), n_classes=N_CLASSES):
    inputs = keras.Input(shape=input_shape)
    x = inputs
    for filtros in [32, 64]:
        x = layers.Conv2D(filtros, (3, 3), padding="same", activation="relu")(x)
        x = layers.BatchNormalization()(x)
        x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Flatten()(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(n_classes, activation="softmax")(x)
    return keras.Model(inputs, outputs, name="CNN_Shallow")

keras.utils.set_random_seed(SEED)
model1 = compilar(build_cnn_shallow(), 1e-3)

hist1 = entrenar(model1, "cnn_shallow")
plot_history(hist1, "CNN Shallow en GPU")
acc1_val = accuracy_score(y_val, predecir(model1, X_val))
print(f"CNN Shallow: accuracy en validación {acc1_val:.4f}")

Epoch 1/30


  1/952 ━━━━━━━━━━━━━━━━━━━━ 1:25:05 5s/step - accuracy: 0.0156 - loss: 5.8410

  5/952 ━━━━━━━━━━━━━━━━━━━━ 13s 15ms/step - accuracy: 0.0406 - loss: 10.2683 

  9/952 ━━━━━━━━━━━━━━━━━━━━ 13s 15ms/step - accuracy: 0.0538 - loss: 7.8227 

 13/952 ━━━━━━━━━━━━━━━━━━━━ 13s 15ms/step - accuracy: 0.0565 - loss: 6.4814

 17/952 ━━━━━━━━━━━━━━━━━━━━ 13s 15ms/step - accuracy: 0.0533 - loss: 5.7542

... (7,196 outputs left out)

952/952 ━━━━━━━━━━━━━━━━━━━━ 13s 14ms/step - accuracy: 0.9917 - loss: 0.0250 - val_accuracy: 0.9991 - val_loss: 0.0030 - learning_rate: 2.5000e-04


Restoring model weights from the end of the best epoch: 30.


[entrenamiento | cnn_shallow | rep 1 | GPU] 411.62 s


CNN Shallow: accuracy en validación 0.9991


### 4.3 Modelo 2: CNN Medium

Modelo de referencia del benchmark: se entrena y se evalúa en test `REPS` veces desde los mismos pesos iniciales, como en la versión secuencial.

In [15]:
def build_cnn_medium(input_shape=(64, 64, 3), n_classes=N_CLASSES, l2_reg=1e-4):
    inputs = keras.Input(shape=input_shape)
    x = inputs
    for i, filtros in enumerate([32, 64, 128, 256], start=1):
        x = layers.Conv2D(filtros, (3, 3), padding="same", activation="relu")(x)
        x = layers.BatchNormalization()(x)
        x = layers.MaxPooling2D((2, 2))(x) if i < 4 else layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(512, activation="relu", kernel_regularizer=regularizers.l2(l2_reg))(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(n_classes, activation="softmax")(x)
    return keras.Model(inputs, outputs, name="CNN_Medium")

for rep in range(1, REPS + 1):
    keras.utils.set_random_seed(SEED)
    red_medium = build_cnn_medium()
    model2 = compilar(red_medium, 5e-4)
    if rep == 1:
        red_medium.summary()
    hist2 = entrenar(model2, "cnn_medium", rep, verbose=1 if rep == 1 else 2)
    y_test_medium = predecir(model2, X_test, "inferencia_test", "cnn_medium", rep)

plot_history(hist2, "CNN Medium en GPU")
acc2_val = accuracy_score(y_val, predecir(model2, X_val))
print(f"CNN Medium: accuracy en validación {acc2_val:.4f} | test {accuracy_score(y_test, y_test_medium):.4f}")

Model: "CNN_Medium"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 64, 64, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 64, 64, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 64, 64, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 32, 32, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 32, 32, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 32, 32, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 16, 16, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 16, 16, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 16, 16, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 8, 8, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 8, 8, 256)      │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 8, 8, 256)      │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 256)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 512)            │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 29)             │        14,877 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 536,797 (2.05 MB)

 Trainable params: 535,837 (2.04 MB)

 Non-trainable params: 960 (3.75 KB)

Epoch 1/40


... (5,417 outputs left out)

Restoring model weights from the end of the best epoch: 9.


[entrenamiento | cnn_medium | rep 1 | GPU] 305.86 s


[inferencia_test | cnn_medium | rep 1 | GPU] 1.65 s


CNN Medium: accuracy en validación 1.0000 | test 1.0000


### 4.4 Selección de la mejor CNN

In [16]:
print(pd.DataFrame([
    {"Modelo": "CNN Shallow (2 bloques)", "Val Accuracy": f"{acc1_val:.4f}",
     "Parámetros": f"{model1.count_params():,}", "Épocas": len(hist1.history["loss"])},
    {"Modelo": "CNN Medium (4 bloques)", "Val Accuracy": f"{acc2_val:.4f}",
     "Parámetros": f"{model2.count_params():,}", "Épocas": len(hist2.history["loss"])},
]).to_string(index=False))

if acc2_val >= acc1_val:
    best_name, best_model, y_test_cnn = "CNN Medium", model2, y_test_medium
else:
    best_name, best_model = "CNN Shallow", model1
    y_test_cnn = predecir(model1, X_test, "inferencia_test", "cnn_shallow")
BEST_KEY = best_name.lower().replace(" ", "_")
print(f"\nMejor CNN: {best_name}")

                 Modelo Val Accuracy Parámetros  Épocas
CNN Shallow (2 bloques)       0.9991  4,221,789      30
 CNN Medium (4 bloques)       1.0000    536,797      17

Mejor CNN: CNN Medium


### 4.5 Red fully-connected

In [17]:
def build_fc_base(input_shape=(64, 64, 3), n_classes=N_CLASSES):
    inputs = keras.Input(shape=input_shape)
    x = layers.Flatten()(inputs)
    for unidades, dropout in [(512, 0.5), (256, 0.4)]:
        x = layers.Dense(unidades, activation="relu")(x)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(dropout)(x)
    outputs = layers.Dense(n_classes, activation="softmax")(x)
    return keras.Model(inputs, outputs, name="FC_Base")

keras.utils.set_random_seed(SEED)
fc1 = compilar(build_fc_base(), 1e-3)

hist_fc1 = entrenar(fc1, "fc_base")
plot_history(hist_fc1, "FC Base en GPU")
acc_fc_val = accuracy_score(y_val, predecir(fc1, X_val))
y_test_fc = predecir(fc1, X_test, "inferencia_test", "fc_base")
print(f"FC Base: accuracy en validación {acc_fc_val:.4f}")

Epoch 1/40


  1/952 ━━━━━━━━━━━━━━━━━━━━ 40:50 3s/step - accuracy: 0.0156 - loss: 4.8568

 16/952 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.0430 - loss: 4.5193  

 31/952 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.0549 - loss: 4.2908

 46/952 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.0622 - loss: 4.1427

 61/952 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.0751 - loss: 4.0070

... (2,590 outputs left out)

952/952 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.7914 - loss: 0.5992 - val_accuracy: 0.9302 - val_loss: 0.2606 - learning_rate: 1.2500e-04


Restoring model weights from the end of the best epoch: 40.


[entrenamiento | fc_base | rep 1 | GPU] 146.85 s


[inferencia_test | fc_base | rep 1 | GPU] 0.43 s
FC Base: accuracy en validación 0.9302


### 4.6 Control: la misma CNN Medium en CPU, desde memoria

La versión CUDA cambia dos cosas a la vez: deja de leer del disco en cada época y mueve el cálculo a la GPU. Para separar ambos efectos, solo en `benchmark` se entrena una vez la CNN Medium con los mismos datos en memoria pero con las operaciones fijadas en el CPU.

In [18]:
if MODO == "benchmark":
    keras.utils.set_random_seed(SEED)
    with tf.device("/CPU:0"):
        control = compilar(build_cnn_medium(), 5e-4)
        hist_control = entrenar(control, "cnn_medium_cpu_memoria", verbose=2, dispositivo="CPU")
    print("Dispositivo de los pesos del control:", control.trainable_variables[0].value.device)
    del control

## 5. Kernels propios con Numba CUDA

La sección 4 delega el reparto del trabajo a cuDNN. Aquí se escribe a mano la inferencia de la CNN Medium para mostrar ese reparto: cada operación de la red es un kernel CUDA y **cada hilo de la GPU calcula un solo elemento de la salida**.

| Kernel | Un hilo calcula | Trabajo por hilo |
|---|---|---|
| `normalizar` | 1 valor de 1 píxel | 1 multiplicación |
| `conv3x3_relu_bn` | 1 píxel de salida de 1 filtro de 1 imagen | 9 × C_in multiplicaciones y sumas, ReLU y BatchNorm |
| `maxpool2x2` | 1 píxel de salida de 1 canal | máximo de 4 valores |
| `global_avg_pool` | 1 canal de 1 imagen | promedio de 8 × 8 valores |
| `dense` | 1 neurona de 1 imagen | producto punto con la fila de pesos |
| `softmax` | 1 imagen | 29 exponenciales y su normalización |

En inferencia, la BatchNorm es una escala y un desplazamiento fijos por canal, así que se aplica dentro del mismo hilo que calcula la convolución. Los pesos son los de la CNN Medium entrenada en la sección 4.

Keras guarda las imágenes como `(N, alto, ancho, canales)` (NHWC). Los kernels trabajan en `(N, canales, alto, ancho)` (NCHW): así los hilos vecinos de un warp, que calculan píxeles vecinos de la misma fila, leen posiciones contiguas de memoria y la GPU agrupa esas lecturas en una sola transacción (acceso coalescente). `normalizar` hace el cambio de formato al convertir a `float32`.

In [19]:
CERO = float32(0.0)

@cuda.jit
def normalizar(x, out):
    j, i, z = cuda.grid(3)                       # columna, fila y (imagen, canal)
    n_img, canales, alto, ancho = out.shape
    if i < alto and j < ancho and z < n_img * canales:
        n, c = z // canales, z % canales
        out[n, c, i, j] = float32(x[n, i, j, c]) * float32(1.0 / 255.0)   # NHWC uint8 a NCHW float32

@cuda.jit(fastmath=True)
def conv3x3_relu_bn(x, w, b, escala, desplazamiento, out):
    j, i, z = cuda.grid(3)                       # columna, fila y (imagen, filtro)
    n_img, filtros, alto, ancho = out.shape
    if i >= alto or j >= ancho or z >= n_img * filtros:
        return
    n, f = z // filtros, z % filtros
    acc = b[f]
    for c in range(x.shape[1]):
        for di in range(3):
            ii = i + di - 1
            if ii < 0 or ii >= alto:             # relleno "same": fuera de la imagen vale 0
                continue
            for dj in range(3):
                jj = j + dj - 1
                if jj >= 0 and jj < ancho:
                    acc += x[n, c, ii, jj] * w[f, c, di, dj]
    out[n, f, i, j] = max(acc, CERO) * escala[f] + desplazamiento[f]

@cuda.jit
def maxpool2x2(x, out):
    j, i, z = cuda.grid(3)
    n_img, canales, alto, ancho = out.shape
    if i < alto and j < ancho and z < n_img * canales:
        n, c = z // canales, z % canales
        out[n, c, i, j] = max(max(x[n, c, 2 * i, 2 * j], x[n, c, 2 * i, 2 * j + 1]),
                              max(x[n, c, 2 * i + 1, 2 * j], x[n, c, 2 * i + 1, 2 * j + 1]))

@cuda.jit(fastmath=True)
def global_avg_pool(x, out):
    c, n = cuda.grid(2)
    if n < out.shape[0] and c < out.shape[1]:
        acc = CERO
        for i in range(x.shape[2]):
            for j in range(x.shape[3]):
                acc += x[n, c, i, j]
        out[n, c] = acc / float32(x.shape[2] * x.shape[3])

@cuda.jit(fastmath=True)
def dense(x, w, b, con_relu, out):
    k, n = cuda.grid(2)                          # neurona e imagen
    if n < out.shape[0] and k < out.shape[1]:
        acc = b[k]
        for c in range(x.shape[1]):
            acc += x[n, c] * w[k, c]
        out[n, k] = max(acc, CERO) if con_relu else acc

@cuda.jit
def softmax(x, out):
    n = cuda.grid(1)
    if n < x.shape[0]:
        maximo = x[n, 0]
        for k in range(1, x.shape[1]):
            maximo = max(maximo, x[n, k])
        suma = CERO
        for k in range(x.shape[1]):
            out[n, k] = math.exp(x[n, k] - maximo)
            suma += out[n, k]
        for k in range(x.shape[1]):
            out[n, k] /= suma

### 5.1 Reparto en hilos, bloques y grid

`config_lanzamiento` elige el bloque y el grid de cada kernel a partir de la forma de su salida:

- **Salidas con forma de imagen** `(N, canales, alto, ancho)`: grid de 3 dimensiones. Los ejes `x` e `y` recorren el ancho y el alto de la imagen, y el eje `z` recorre los `N × canales` mapas de salida. El bloque es de hasta 16 × 16 hilos; si la imagen es más pequeña (8 × 8), el bloque crece en `z` para seguir teniendo 256 hilos.
- **Salidas con forma de matriz** `(N, unidades)`: grid de 2 dimensiones con bloques de 16 × 16.
- **Softmax**: grid de 1 dimensión, un hilo por imagen, en bloques de 32.

CUDA admite 65,535 bloques como máximo en el eje `z` del grid, lo que limita `N × canales`. Con lotes de 64 imágenes el caso más grande queda muy por debajo.

In [20]:
HILOS_POR_BLOQUE = 256

def config_lanzamiento(forma):
    """Devuelve (grid, bloque) para que haya al menos un hilo por elemento de la salida."""
    if len(forma) == 4:
        n, canales, alto, ancho = forma
        bx, by = min(16, ancho), min(16, alto)
        bz = max(1, HILOS_POR_BLOQUE // (bx * by))
        grid = (math.ceil(ancho / bx), math.ceil(alto / by), math.ceil(n * canales / bz))
        assert grid[2] <= 65535, "demasiados bloques en el eje z: reduzca el lote"
        return grid, (bx, by, bz)
    n, unidades = forma
    return (math.ceil(unidades / 16), math.ceil(n / 16)), (16, 16)

def extraer_pesos(modelo):
    """Pesos de Keras reordenados para los kernels, con la BatchNorm reducida a escala y desplazamiento."""
    convs = [c for c in modelo.layers if isinstance(c, layers.Conv2D)]
    norms = [c for c in modelo.layers if isinstance(c, layers.BatchNormalization)]
    densas = [c for c in modelo.layers if isinstance(c, layers.Dense)]
    bloques = []
    for conv, bn in zip(convs, norms):
        w, b = conv.get_weights()                          # w: (3, 3, C_in, C_out)
        gamma, beta, media, varianza = bn.get_weights()
        escala = gamma / np.sqrt(varianza + bn.epsilon)
        bloques.append((w.transpose(3, 2, 0, 1), b, escala, beta - media * escala))
    return bloques, [(w.T, b) for w, b in (d.get_weights() for d in densas)]

def a_gpu(arreglo):
    return cuda.to_device(np.ascontiguousarray(arreglo, dtype=np.float32))

class RedNumba:
    """Inferencia de la CNN Medium con kernels propios, por lotes de tamaño fijo."""

    def __init__(self, modelo, lote=BATCH_SIZE):
        bloques, densas = extraer_pesos(modelo)
        self.lote = lote
        self.capas = []
        self.entrada = cuda.device_array((lote, *TARGET_SIZE, 3), dtype=np.uint8)
        x = self._capa("normalizar", normalizar, (self.entrada,), (lote, 3, *TARGET_SIZE))
        lado = TARGET_SIZE[0]
        for k, pesos in enumerate(bloques, start=1):
            filtros = pesos[0].shape[0]
            x = self._capa(f"conv{k}", conv3x3_relu_bn, (x, *map(a_gpu, pesos)), (lote, filtros, lado, lado))
            if k < len(bloques):
                lado //= 2
                x = self._capa(f"maxpool{k}", maxpool2x2, (x,), (lote, filtros, lado, lado))
            else:
                x = self._capa("global_avg_pool", global_avg_pool, (x,), (lote, filtros))
        for k, (w, b) in enumerate(densas, start=1):
            x = self._capa(f"dense{k}", dense, (x, a_gpu(w), a_gpu(b), k < len(densas)), (lote, w.shape[0]))
        self.salida = self._capa("softmax", softmax, (x,), (lote, N_CLASSES), ((math.ceil(lote / 32),), (32,)))

    def _capa(self, nombre, kernel, argumentos, forma, config=None):
        salida = cuda.device_array(forma, dtype=np.float32)
        grid, bloque = config or config_lanzamiento(forma)
        self.capas.append({"nombre": nombre, "kernel": kernel[grid, bloque], "args": (*argumentos, salida),
                           "forma": forma, "grid": grid, "bloque": bloque})
        return salida

    def forward(self, tiempos=None):
        for capa in self.capas:
            t0 = time.perf_counter()
            capa["kernel"](*capa["args"])
            if tiempos is not None:
                cuda.synchronize()
                tiempos[capa["nombre"]] = tiempos.get(capa["nombre"], 0.0) + time.perf_counter() - t0

    def predecir(self, X):
        """Devuelve las probabilidades y los segundos gastados en H2D, kernels y D2H."""
        probs = np.empty((len(X), N_CLASSES), dtype=np.float32)
        lote_cpu = np.zeros(self.entrada.shape, dtype=np.uint8)
        salida_cpu = np.empty((self.lote, N_CLASSES), dtype=np.float32)
        t = {"h2d": 0.0, "kernels": 0.0, "d2h": 0.0}
        for i in range(0, len(X), self.lote):
            n = min(self.lote, len(X) - i)
            lote_cpu[:n] = X[i:i + n]
            t0 = time.perf_counter()
            self.entrada.copy_to_device(lote_cpu)
            cuda.synchronize()
            t1 = time.perf_counter()
            self.forward()
            cuda.synchronize()
            t2 = time.perf_counter()
            self.salida.copy_to_host(salida_cpu)
            t3 = time.perf_counter()
            probs[i:i + n] = salida_cpu[:n]
            t["h2d"] += t1 - t0
            t["kernels"] += t2 - t1
            t["d2h"] += t3 - t2
        return probs, t

### 5.2 Compilación y configuración de lanzamiento de cada capa

La primera llamada compila los kernels a código de GPU, y ese tiempo se registra aparte. Después se ejecutan 10 lotes sincronizando tras cada kernel para medir cuánto tarda cada capa.

In [21]:
with cronometro("compilacion_jit", "cnn_medium_numba", 1):
    red_numba = RedNumba(red_medium)
    red_numba.predecir(X_test[:BATCH_SIZE])

LOTES_PERFIL = 10
tiempos_capa = {}
for _ in range(LOTES_PERFIL):
    red_numba.forward(tiempos_capa)

lanzamiento = pd.DataFrame([{
    "Capa": c["nombre"],
    "Salida": " x ".join(map(str, c["forma"])),
    "Grid": " x ".join(map(str, c["grid"])),
    "Bloque": " x ".join(map(str, c["bloque"])),
    "Bloques": math.prod(c["grid"]),
    "Hilos": math.prod(c["grid"]) * math.prod(c["bloque"]),
    "Elementos de salida": math.prod(c["forma"]) if c["nombre"] != "softmax" else c["forma"][0],
    "ms por lote": 1000 * tiempos_capa[c["nombre"]] / LOTES_PERFIL,
} for c in red_numba.capas])
lanzamiento.to_csv(RESULTS_DIR / f"kernels_numba_{MODO}.csv", index=False)
print(f"Lote de {BATCH_SIZE} imágenes")
print(lanzamiento.to_string(index=False, float_format=lambda v: f"{v:.3f}",
                            formatters={c: "{:,}".format for c in ["Bloques", "Hilos", "Elementos de salida"]}))

[compilacion_jit | cnn_medium_numba | rep 1 | GPU] 0.51 s


Lote de 64 imágenes
           Capa             Salida         Grid      Bloque Bloques     Hilos Elementos de salida  ms por lote
     normalizar   64 x 3 x 64 x 64  4 x 4 x 192 16 x 16 x 1   3,072   786,432             786,432        0.105
          conv1  64 x 32 x 64 x 64 4 x 4 x 2048 16 x 16 x 1  32,768 8,388,608           8,388,608        2.145
       maxpool1  64 x 32 x 32 x 32 2 x 2 x 2048 16 x 16 x 1   8,192 2,097,152           2,097,152        0.278
          conv2  64 x 64 x 32 x 32 2 x 2 x 4096 16 x 16 x 1  16,384 4,194,304           4,194,304        9.485
       maxpool2  64 x 64 x 16 x 16 1 x 1 x 4096 16 x 16 x 1   4,096 1,048,576           1,048,576        0.168
          conv3 64 x 128 x 16 x 16 1 x 1 x 8192 16 x 16 x 1   8,192 2,097,152           2,097,152        8.257
       maxpool3   64 x 128 x 8 x 8 1 x 1 x 2048   8 x 8 x 4   2,048   524,288             524,288        0.106
          conv4   64 x 256 x 8 x 8 1 x 1 x 4096   8 x 8 x 4   4,096 1,048,576           1,04

### 5.3 Inferencia en test y validación contra Keras

Las 13,050 imágenes de test pasan por los kernels en lotes de 64. Se mide por separado la copia del lote a la GPU (H2D), la ejecución de los kernels y la copia de las probabilidades de vuelta (D2H). El resultado se compara con el de Keras para el mismo modelo.

In [22]:
for rep in range(1, REPS + 1):
    with cronometro("inferencia_test", "cnn_medium_numba", rep, len(X_test)):
        probs_numba, partes = red_numba.predecir(X_test)
    for parte, segundos in partes.items():
        registrar(f"numba_{parte}", "cnn_medium_numba", rep, segundos, len(X_test))

probs_keras = model2.predict(X_test, batch_size=BATCH_SIZE, verbose=0)
y_test_numba = probs_numba.argmax(axis=1)
coincidencias = int((y_test_numba == probs_keras.argmax(axis=1)).sum())
dif_max = float(np.abs(probs_numba - probs_keras).max())
acc_numba = accuracy_score(y_test, y_test_numba)

print(f"\nPredicciones iguales a las de Keras: {coincidencias:,} de {len(X_test):,}")
print(f"Diferencia máxima entre probabilidades: {dif_max:.2e}")
print(f"Accuracy en test: kernels Numba {acc_numba:.4f} | Keras en GPU {accuracy_score(y_test, y_test_medium):.4f}")
print("Reparto del tiempo: " + " | ".join(f"{k} {v:.2f} s ({100 * v / sum(partes.values()):.0f} %)" for k, v in partes.items()))
assert coincidencias >= 0.999 * len(X_test) and dif_max < 1e-3, "los kernels no reproducen la salida de Keras"

[inferencia_test | cnn_medium_numba | rep 1 | GPU] 6.13 s



Predicciones iguales a las de Keras: 13,050 de 13,050
Diferencia máxima entre probabilidades: 8.94e-07
Accuracy en test: kernels Numba 1.0000 | Keras en GPU 1.0000
Reparto del tiempo: h2d 0.06 s (1 %) | kernels 6.04 s (99 %) | d2h 0.01 s (0 %)


## 6. Modelo clásico en CPU multinúcleo

El modelo clásico no es una red neuronal: son árboles de decisión sobre características HOG, con muchas ramas y accesos irregulares a memoria. No se lleva a la GPU. Se paraleliza en el CPU con `N_JOBS` procesos para la extracción de características y `N_JOBS` hilos para los ensambles. El flujo es el de la versión secuencial.

In [23]:
HOG_PARAMS = dict(orientations=9, pixels_per_cell=(8, 8), cells_per_block=(2, 2),
                  block_norm="L2-Hys", feature_vector=True)

def a_arreglo(im, size, modo):
    return np.asarray(im.convert(modo).resize(size, Image.BICUBIC), dtype=np.float32) / 255.0

def feat_raw_gray32(im):
    return a_arreglo(im, (32, 32), "L").ravel()

def feat_hog_color(im):
    h = hog(a_arreglo(im, TARGET_SIZE, "L"), **HOG_PARAMS).astype(np.float32)
    return np.concatenate([h, a_arreglo(im, (8, 8), "RGB").ravel()])

FEATURES = {"raw_gray32": feat_raw_gray32, "hog+color": feat_hog_color}

def extraer(path, feat_fn):
    with Image.open(path) as im:
        return feat_fn(im)

def build_matrix(df, feat_fn):
    X = np.asarray(Parallel(n_jobs=N_JOBS, batch_size=128)(delayed(extraer)(p, feat_fn) for p in df["path"]),
                   dtype=np.float32)
    return X, df["clase"].map(label_map).to_numpy()

def subsample(df, n_per_class):
    idx = df.groupby("clase")["path"].sample(n=n_per_class, random_state=SEED).index
    return df.loc[idx].reset_index(drop=True)

def candidatos():
    return {
        "KNN(k=3)": KNeighborsClassifier(n_neighbors=3, n_jobs=N_JOBS),
        "RF(300)": RandomForestClassifier(n_estimators=300, n_jobs=N_JOBS, random_state=SEED),
        "ExtraTrees(300)": ExtraTreesClassifier(n_estimators=300, n_jobs=N_JOBS, random_state=SEED),
        "SVC-RBF": make_pipeline(StandardScaler(), SVC(C=10, gamma="scale", random_state=SEED)),
    }

bench_train, bench_val = subsample(train_df, 150), subsample(val_df, 75)
filas = []
with cronometro("barrido", "clasico", 1, len(bench_train) + len(bench_val), CPU_N):
    for f_nombre, fn in FEATURES.items():
        Xtr, ytr = build_matrix(bench_train, fn)
        Xva, yva = build_matrix(bench_val, fn)
        for m_nombre, modelo in candidatos().items():
            t0 = time.perf_counter()
            modelo.fit(Xtr, ytr)
            t_fit = time.perf_counter() - t0
            filas.append({"Representación": f_nombre, "Modelo": m_nombre,
                          "Val Accuracy": accuracy_score(yva, modelo.predict(Xva)), "Fit (s)": t_fit})

print(pd.DataFrame(filas).sort_values("Val Accuracy", ascending=False)
      .to_string(index=False, float_format=lambda v: f"{v:.4f}"))

[barrido | clasico | rep 1 | CPU x16] 39.45 s
Representación          Modelo  Val Accuracy  Fit (s)
    raw_gray32 ExtraTrees(300)        0.9398   0.9060
     hog+color         RF(300)        0.9274   4.5967
    raw_gray32         RF(300)        0.9269   1.8796
     hog+color ExtraTrees(300)        0.9168   1.9724
     hog+color         SVC-RBF        0.9140   9.6100
     hog+color        KNN(k=3)        0.7894   0.0032
    raw_gray32         SVC-RBF        0.7811   3.7828
    raw_gray32        KNN(k=3)        0.6538   0.0024


In [24]:
matrices = {}
for nombre, fn in FEATURES.items():
    with cronometro("extraccion_features", nombre, 1, len(train_fit_df) + len(val_df), CPU_N):
        matrices[nombre] = (*build_matrix(train_fit_df, fn), *build_matrix(val_df, fn))

filas = []
with cronometro("ajuste_hiperparametros", "clasico", 1, len(train_fit_df), CPU_N):
    for nombre, (Xtr, ytr, Xva, yva) in matrices.items():
        for n in [50, 100, 200, 300]:
            modelo = ExtraTreesClassifier(n_estimators=n, n_jobs=N_JOBS, random_state=SEED)
            t0 = time.perf_counter()
            modelo.fit(Xtr, ytr)
            t_fit = time.perf_counter() - t0
            filas.append({"Representación": nombre, "n_estimators": n,
                          "Val Accuracy": accuracy_score(yva, modelo.predict(Xva)), "Fit (s)": t_fit})
            del modelo

ajuste = pd.DataFrame(filas)
print(ajuste.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

elegibles = ajuste[ajuste["Val Accuracy"] >= ajuste["Val Accuracy"].max() - 0.001]
elegida = elegibles.sort_values(["n_estimators", "Val Accuracy"], ascending=[True, False]).iloc[0]
FEAT_ELEGIDA, N_ESTIMATORS = elegida["Representación"], int(elegida["n_estimators"])
MODELO_ELEGIDO = f"ExtraTrees({N_ESTIMATORS}) + {FEAT_ELEGIDA}"
print(f"\nConfiguración elegida: {MODELO_ELEGIDO} | val accuracy: {elegida['Val Accuracy']:.4f}")

X_train_c, y_train_c = matrices[FEAT_ELEGIDA][:2]
del matrices
modelo_final = ExtraTreesClassifier(n_estimators=N_ESTIMATORS, n_jobs=N_JOBS, random_state=SEED)
with cronometro("entrenamiento", "clasico", 1, len(X_train_c), CPU_N):
    modelo_final.fit(X_train_c, y_train_c)

with cronometro("extraccion_features_test", "clasico", 1, len(test_df), CPU_N):
    X_test_c, y_test_c = build_matrix(test_df, FEATURES[FEAT_ELEGIDA])
with cronometro("inferencia_test", "clasico", 1, len(test_df), CPU_N):
    y_pred_c = modelo_final.predict(X_test_c)
print(f"{MODELO_ELEGIDO}: accuracy en test {accuracy_score(y_test_c, y_pred_c):.4f}")

[extraccion_features | raw_gray32 | rep 1 | CPU x16] 4.77 s


[extraccion_features | hog+color | rep 1 | CPU x16] 17.03 s


[ajuste_hiperparametros | clasico | rep 1 | CPU x16] 273.41 s
Representación  n_estimators  Val Accuracy  Fit (s)
    raw_gray32            50        0.9959   8.0025
    raw_gray32           100        0.9966  15.7345
    raw_gray32           200        0.9972  31.5102
    raw_gray32           300        0.9971  47.0007
     hog+color            50        0.9969  12.8538
     hog+color           100        0.9982  25.5517
     hog+color           200        0.9986  51.5034
     hog+color           300        0.9989  77.5567

Configuración elegida: ExtraTrees(100) + hog+color | val accuracy: 0.9982


[entrenamiento | clasico | rep 1 | CPU x16] 25.52 s


[extraccion_features_test | clasico | rep 1 | CPU x16] 3.14 s


[inferencia_test | clasico | rep 1 | CPU x16] 0.32 s
ExtraTrees(100) + hog+color: accuracy en test 0.9986


## 7. Exactitud frente a la versión secuencial

Acelerar no sirve si el resultado cambia. Se comparan las exactitudes de esta corrida con las que dejó el notebook secuencial en `results/metricas_secuencial_<modo>.json`. No se esperan valores idénticos: el orden de los lotes y el redondeo de las operaciones en GPU son distintos.

In [25]:
FOTOS_DIR = ROOT / "fotos_grupo"
patron = re.compile(r"^([A-Za-z]+)_([A-Za-z0-9]+)")
fotos = [(ruta, m.group(1).upper())
         for ruta in sorted(p for ext in ("*.jpg", "*.jpeg", "*.png") for p in FOTOS_DIR.glob(ext))
         if (m := patron.match(ruta.stem))]

def recorte_cuadrado(ruta):
    with Image.open(ruta) as im:
        im = im.convert("RGB")
        lado = min(im.size)
        izq, arriba = (im.width - lado) // 2, (im.height - lado) // 2
        return im.crop((izq, arriba, izq + lado, arriba + lado))

X_fotos = np.stack([np.asarray(recorte_cuadrado(r).resize(TARGET_SIZE, Image.BICUBIC), dtype=np.uint8) for r, _ in fotos])
pred_fotos = [idx_to_class[i] for i in best_model.predict(X_fotos, verbose=0).argmax(axis=1)]
n_correctos = sum(p == real for p, (_, real) in zip(pred_fotos, fotos))

metricas = {
    "val": {"cnn_shallow": float(acc1_val), "cnn_medium": float(acc2_val), "fc_base": float(acc_fc_val),
            "clasico": float(elegida["Val Accuracy"])},
    "test": {BEST_KEY: float(accuracy_score(y_test, y_test_cnn)), "cnn_medium_numba": float(acc_numba),
             "fc_base": float(accuracy_score(y_test, y_test_fc)), "clasico": float(accuracy_score(y_test_c, y_pred_c))},
    "epocas": {"cnn_shallow": len(hist1.history["loss"]), "cnn_medium": len(hist2.history["loss"]),
               "fc_base": len(hist_fc1.history["loss"])},
    "mejor_cnn": BEST_KEY, "modelo_clasico": MODELO_ELEGIDO,
    "fotos_propias": f"{n_correctos}/{len(fotos)}",
}
(RESULTS_DIR / f"metricas_cuda_{MODO}.json").write_text(json.dumps(metricas, indent=2, ensure_ascii=False))

JSON_SEC = RESULTS_DIR / f"metricas_secuencial_{MODO}.json"
metricas_sec = json.loads(JSON_SEC.read_text()) if JSON_SEC.exists() else {}
filas = []
for conjunto in ("val", "test"):
    for modelo, valor in metricas[conjunto].items():
        sec = metricas_sec.get(conjunto, {}).get(modelo.replace("_numba", ""))
        filas.append({"Conjunto": conjunto, "Modelo": modelo, "Secuencial": np.nan if sec is None else sec,
                      "CUDA": valor, "Diferencia": np.nan if sec is None else valor - sec})
exactitud = pd.DataFrame(filas)
exactitud.to_csv(RESULTS_DIR / f"exactitud_{MODO}.csv", index=False)
print(exactitud.to_string(index=False, float_format=lambda v: f"{v:.4f}", na_rep="sin dato"))
print(f"\nFotos propias con {best_name}: {n_correctos}/{len(fotos)} aciertos"
      + (f" (secuencial: {metricas_sec['fotos_propias']})" if metricas_sec else ""))

Conjunto           Modelo  Secuencial   CUDA  Diferencia
     val      cnn_shallow      0.9994 0.9991     -0.0003
     val       cnn_medium      1.0000 1.0000      0.0000
     val          fc_base      0.9238 0.9302      0.0064
     val          clasico      0.9982 0.9982      0.0000
    test       cnn_medium      1.0000 1.0000      0.0000
    test cnn_medium_numba      1.0000 1.0000      0.0000
    test          fc_base      0.9265 0.9303      0.0038
    test          clasico      0.9986 0.9986      0.0000

Fotos propias con CNN Medium: 4/15 aciertos (secuencial: 4/15)


## 8. Comparación de tiempos y speedup

Speedup = tiempo secuencial / tiempo de la versión CUDA, por fase, con la media de las repeticiones. Para leer la tabla:

- **Época y entrenamiento.** El tiempo secuencial incluye leer y decodificar las imágenes del disco en cada época. En la versión CUDA ese trabajo se hizo una sola vez en la fase 1, así que la fila *Pipeline completo* suma preparación, entrenamiento e inferencia para comparar de extremo a extremo.
- **Primera época en GPU.** Incluye la inicialización de cuDNN y la compilación del grafo, por eso es más lenta que las siguientes. Se promedia junto con las demás.
- **Modo completo.** Con EarlyStopping el número de épocas puede diferir entre versiones. En ese modo, el tiempo por época es la comparación directa.

In [26]:
registrar("notebook_completo", "-", 1, time.perf_counter() - T_INICIO_NOTEBOOK, dispositivo="CPU y GPU")
tiempos = pd.DataFrame(REGISTRO_TIEMPOS)

resumen = (tiempos[tiempos["fase"] != "epoca"]
           .groupby(["fase", "modelo", "dispositivo"], sort=False)
           .agg(repeticiones=("segundos", "size"), media_s=("segundos", "mean"),
                desv_s=("segundos", "std"), n_imagenes=("n_imagenes", "first"))
           .reset_index())
resumen["ms_por_imagen"] = 1000 * resumen["media_s"] / resumen["n_imagenes"]
resumen.to_csv(RESULTS_DIR / f"resumen_cuda_{MODO}.csv", index=False)

por_epoca = (tiempos[tiempos["fase"] == "epoca"].groupby("modelo", sort=False)["segundos"]
             .agg(epocas="size", media_s="mean", desv_s="std", primera_s="first", mediana_s="median").reset_index())
por_epoca.to_csv(RESULTS_DIR / f"tiempo_por_epoca_cuda_{MODO}.csv", index=False)

print(resumen.round(3).to_string(index=False))
print()
print(por_epoca.round(3).to_string(index=False))

                    fase            modelo dispositivo  repeticiones  media_s  desv_s  n_imagenes  ms_por_imagen
              inventario                 -         CPU             1    0.779     NaN         NaN            NaN
               particion                 -         CPU             1    0.532     NaN         NaN            NaN
           arranque_pool          paralelo     CPU x16             1    0.316     NaN         NaN            NaN
        preprocesamiento          paralelo     CPU x16             1    5.087     NaN     60900.0          0.084
        preprocesamiento               val     CPU x16             1    1.109     NaN     13050.0          0.085
        preprocesamiento              test     CPU x16             1    1.117     NaN     13050.0          0.086
       transferencia_h2d       train_uint8         GPU             1    0.277     NaN     60900.0          0.005
       transferencia_d2h predicciones_test         GPU             1    0.055     NaN     13050.

In [27]:
CSV_SEC = RESULTS_DIR / f"tiempos_secuencial_{MODO}.csv"
HAY_SECUENCIAL = CSV_SEC.exists()

PARES = [  # fila de la tabla, (fase, modelo) en la versión secuencial, (fase, modelo) en la versión CUDA
    ("Preprocesamiento de train", ("preprocesamiento", "secuencial"), ("preprocesamiento", "paralelo")),
    ("Época: CNN Shallow", ("epoca", "cnn_shallow"), ("epoca", "cnn_shallow")),
    ("Época: CNN Medium", ("epoca", "cnn_medium"), ("epoca", "cnn_medium")),
    ("Época: FC Base", ("epoca", "fc_base"), ("epoca", "fc_base")),
    ("Entrenamiento: CNN Shallow", ("entrenamiento", "cnn_shallow"), ("entrenamiento", "cnn_shallow")),
    ("Entrenamiento: CNN Medium", ("entrenamiento", "cnn_medium"), ("entrenamiento", "cnn_medium")),
    ("Entrenamiento: FC Base", ("entrenamiento", "fc_base"), ("entrenamiento", "fc_base")),
    ("Inferencia en test: CNN Medium, Keras", ("inferencia_test", "cnn_medium"), ("inferencia_test", "cnn_medium")),
    ("Inferencia en test: CNN Medium, kernels Numba", ("inferencia_test", "cnn_medium"), ("inferencia_test", "cnn_medium_numba")),
    ("Inferencia en test: FC Base", ("inferencia_test", "fc_base"), ("inferencia_test", "fc_base")),
    ("Clásico: barrido de familias", ("barrido", "clasico"), ("barrido", "clasico")),
    ("Clásico: extracción raw_gray32", ("extraccion_features", "raw_gray32"), ("extraccion_features", "raw_gray32")),
    ("Clásico: extracción hog+color", ("extraccion_features", "hog+color"), ("extraccion_features", "hog+color")),
    ("Clásico: ajuste del número de árboles", ("ajuste_hiperparametros", "clasico"), ("ajuste_hiperparametros", "clasico")),
    ("Clásico: entrenamiento final", ("entrenamiento", "clasico"), ("entrenamiento", "clasico")),
    ("Clásico: extracción de test", ("extraccion_features_test", "clasico"), ("extraccion_features_test", "clasico")),
    ("Clásico: inferencia en test", ("inferencia_test", "clasico"), ("inferencia_test", "clasico")),
]

def medir(df, clave):
    sel = df[(df["fase"] == clave[0]) & (df["modelo"] == clave[1])]
    if sel.empty:
        return np.nan, np.nan, ""
    return sel["segundos"].mean(), sel["segundos"].std(), sel["dispositivo"].iloc[0]

if HAY_SECUENCIAL:
    tiempos_sec = pd.read_csv(CSV_SEC)
    hw_sec = json.loads((RESULTS_DIR / f"hardware_secuencial_{MODO}.json").read_text())
    for campo in ("cpu", "imgs_por_clase_benchmark", "epocas_benchmark", "repeticiones"):
        if hw_sec.get(campo) != HARDWARE.get(campo):
            print(f"AVISO: '{campo}' difiere entre corridas: secuencial {hw_sec.get(campo)!r}, CUDA {HARDWARE.get(campo)!r}")

    filas = []
    for etiqueta, clave_sec, clave_cuda in PARES:
        s_media, s_desv, s_disp = medir(tiempos_sec, clave_sec)
        c_media, c_desv, c_disp = medir(tiempos, clave_cuda)
        if not (np.isnan(s_media) or np.isnan(c_media)):
            filas.append({"fase": etiqueta, "dispositivo_secuencial": s_disp, "secuencial_s": s_media,
                          "secuencial_desv": s_desv, "dispositivo_cuda": c_disp, "cuda_s": c_media, "cuda_desv": c_desv})

    # De extremo a extremo para la CNN Medium: preparación, entrenamiento e inferencia en test
    sec_total = medir(tiempos_sec, ("entrenamiento", "cnn_medium"))[0] + medir(tiempos_sec, ("inferencia_test", "cnn_medium"))[0]
    cuda_total = sum(medir(tiempos, clave)[0] for clave in [
        ("arranque_pool", "paralelo"), ("preprocesamiento", "paralelo"), ("preprocesamiento", "val"),
        ("preprocesamiento", "test"), ("entrenamiento", "cnn_medium"), ("inferencia_test", "cnn_medium")])
    filas.append({"fase": "Pipeline completo: CNN Medium", "dispositivo_secuencial": "CPU", "secuencial_s": sec_total,
                  "dispositivo_cuda": f"{CPU_N} y GPU", "cuda_s": cuda_total})

    comparacion = pd.DataFrame(filas)
    comparacion["speedup"] = comparacion["secuencial_s"] / comparacion["cuda_s"]
    comparacion.to_csv(RESULTS_DIR / f"comparacion_{MODO}.csv", index=False)
    print(comparacion.to_string(index=False, na_rep="", float_format=lambda v: f"{v:,.2f}"))
else:
    print(f"No existe {CSV_SEC.name}: ejecute primero el notebook secuencial en modo {MODO} para calcular el speedup.")

                                         fase dispositivo_secuencial  secuencial_s  secuencial_desv dispositivo_cuda  cuda_s  cuda_desv  speedup
                    Preprocesamiento de train                    CPU         33.41                           CPU x16    5.09                6.57
                           Época: CNN Shallow                    CPU        386.45            27.71              GPU   13.69       2.10    28.23
                            Época: CNN Medium                    CPU        497.53             5.27              GPU   17.95       3.07    27.72
                               Época: FC Base                    CPU        130.66             2.09              GPU    3.64       0.78    35.86
                   Entrenamiento: CNN Shallow                    CPU     11,593.69                               GPU  411.62               28.17
                    Entrenamiento: CNN Medium                    CPU      7,960.59                               GPU  305.86      

In [28]:
if HAY_SECUENCIAL:
    graf = comparacion.iloc[::-1].reset_index(drop=True)
    y = np.arange(len(graf))
    alto = 0.36 * len(graf) + 1.4

    fig, ax = plt.subplots(figsize=(10.5, alto))
    ax.hlines(y, graf["cuda_s"], graf["secuencial_s"], color="#c3c2b7", linewidth=1.5, zorder=1)
    ax.scatter(graf["secuencial_s"], y, s=64, color=AZUL, label="Secuencial", zorder=2, edgecolors=FONDO, linewidths=1.5)
    ax.scatter(graf["cuda_s"], y, s=64, color=NARANJA, label="CUDA", zorder=2, edgecolors=FONDO, linewidths=1.5)
    ax.set_xscale("log")
    ax.set_yticks(y, graf["fase"])
    ax.set_xlabel("Tiempo medio (s, escala logarítmica)")
    ax.set_title(f"Tiempo por fase: versión secuencial y versión CUDA (modo {MODO})")
    ax.grid(True, axis="x")
    ax.legend(loc="lower right", frameon=False)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / f"comparacion_tiempos_{MODO}.png", dpi=150, bbox_inches="tight")
    plt.show()

    fig, ax = plt.subplots(figsize=(10.5, alto))
    ax.barh(y, graf["speedup"], height=0.55, color=AZUL)
    ax.axvline(1, color=TINTA2, linewidth=1)
    for yi, s in zip(y, graf["speedup"]):
        ax.text(s, yi, f" {s:.1f}x", va="center", ha="left", color=TINTA2)
    ax.set_xlim(0, graf["speedup"].max() * 1.12)
    ax.set_yticks(y, graf["fase"])
    ax.set_xlabel("Speedup (tiempo secuencial / tiempo CUDA); la línea marca 1x")
    ax.set_title(f"Speedup por fase (modo {MODO})")
    ax.grid(True, axis="x")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / f"speedup_{MODO}.png", dpi=150, bbox_inches="tight")
    plt.show()

### 8.1 De dónde viene el speedup del entrenamiento

Solo en `benchmark`. Tres formas de entrenar la misma CNN Medium con los mismos datos: como en el laboratorio (CPU, leyendo del disco en cada época), en CPU con los datos ya en memoria, y en GPU con los datos en memoria. La diferencia entre las dos primeras es lo que aporta preparar los datos una sola vez; la diferencia entre las dos últimas es lo que aporta la GPU.

In [29]:
if HAY_SECUENCIAL and MODO == "benchmark":
    etapas = pd.DataFrame([
        {"configuracion": "CPU, lee del disco en cada época (secuencial)", "segundos": medir(tiempos_sec, ("entrenamiento", "cnn_medium"))[0]},
        {"configuracion": "CPU, datos en memoria (control)", "segundos": medir(tiempos, ("entrenamiento", "cnn_medium_cpu_memoria"))[0]},
        {"configuracion": "GPU, datos en memoria (CUDA)", "segundos": medir(tiempos, ("entrenamiento", "cnn_medium"))[0]},
    ])
    etapas["speedup_acumulado"] = etapas["segundos"].iloc[0] / etapas["segundos"]
    etapas.to_csv(RESULTS_DIR / "origen_speedup_benchmark.csv", index=False)
    print(etapas.to_string(index=False, float_format=lambda v: f"{v:,.2f}"))

    fig, ax = plt.subplots(figsize=(9, 2.8))
    y = np.arange(len(etapas))[::-1]
    ax.barh(y, etapas["segundos"], height=0.5, color=AZUL)
    for yi, s, sp in zip(y, etapas["segundos"], etapas["speedup_acumulado"]):
        ax.text(s, yi, f" {s:,.1f} s ({sp:.1f}x)", va="center", ha="left", color=TINTA2)
    ax.set_xlim(0, etapas["segundos"].max() * 1.22)
    ax.set_yticks(y, etapas["configuracion"])
    ax.set_xlabel(f"Tiempo de entrenamiento de la CNN Medium, {EPOCAS_BENCHMARK} épocas (s)")
    ax.set_title("Origen del speedup del entrenamiento")
    ax.grid(True, axis="x")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "origen_speedup_benchmark.png", dpi=150, bbox_inches="tight")
    plt.show()

### 8.2 Uso de recursos

Resumen por fase del log de `MonitorRecursos`. `nucleos_equivalentes` es la carga total expresada en núcleos al 100 % y `nucleos_activos_medio` es el número medio de núcleos lógicos por encima del 50 % de uso. La gráfica se guarda en `evidencias/recursos_cuda_<modo>.png`.

In [30]:
MONITOR.detener()
print(MONITOR.resumen().to_string(index=False))
MONITOR.graficar()
plt.show()

print(f"Tiempo total del notebook: {tiempos.loc[tiempos['fase'] == 'notebook_completo', 'segundos'].iloc[-1] / 60:.1f} min")

                              fase  segundos  cpu_total_medio  nucleos_equivalentes  nucleos_activos_medio  nucleo_mas_cargado_medio  ram_max_gb  gpu_util_media  gpu_util_max  gpu_mem_max_mb
                         particion       1.0              8.3                   1.3                    1.0                     100.0         6.8             0.0           0.0            56.0
       preprocesamiento | paralelo       5.0             88.2                  14.1                   13.0                      99.4         8.2             0.0           0.0            56.0
            preprocesamiento | val       1.0             74.9                  12.0                   16.0                      84.0         9.1             0.0           0.0            56.0
           preprocesamiento | test       1.0             90.6                  14.5                   16.0                     100.0         9.3             0.0           0.0            56.0
       entrenamiento | cnn_shallow     411.0 

Tiempo total del notebook: 20.9 min
